# **VII. Procesamiento de Variables Objetivo y Creación de Variables Derivadas**

## Objetivo
Este notebook tiene como propósito ejecutar la limpieza definitiva, transformación y discretización de las variables objetivo (*targets*) del estudio: Mortalidad, Severidad y Consumo de Recursos. Adicionalmente, consolida la Ingeniería de Características (*Feature Engineering*) mediante la creación de variables derivadas esenciales (Edad, Días de Estadía, Cargas Oncológicas y de Comorbilidades) sin comprometer la integridad matemática del conjunto de datos original ni incurrir en fuga de información (*Data Leakage*).

## Proceso
1. **Aislamiento de Datos**: Generación de copias de seguridad de los datos clasificados hacia un entorno de `Datos procesados`.
2. **Depuración de Targets**: Eliminación estricta de registros con variables objetivo nulas o desconocidas (evitando la imputación de *Ground Truth*).
3. **Binarización y Discretización**: 
   - Conversión de `TIPOALTA` a variable dicotómica (`MORTALIDAD`: 1/0).
   - Estandarización de la escala ordinal de `SEVERIDAD` (0 a 3).
   - Transformación del peso relativo GRD continuo en terciles de `CONSUMO_RECURSOS` (0: Bajo, 1: Medio, 2: Alto) calculados sobre la población total.
4. **Ingeniería de Características**: Cálculo de `EDAD_INGRESO` y `DIAS_ESTADIA` a partir de variables *datetime*; y binarización de `USO_PABELLON`.
5. **Síntesis Clínica**: Vectorización de los 35 campos de diagnóstico para calcular la `CARGA_ONCOLOGICA` y el `NUM_COMORBILIDADES`.

In [1]:
# ============================================================================
# CONFIGURACIÓN INICIAL: Imports y Rutas
# ============================================================================
import pandas as pd
import os
import numpy as np

# Rutas actualizadas a los datos CLASIFICADOS (que incluyen al grupo control)
carpeta_origen = "../../Datos/Datos clasificados"
carpeta_destino = "../../Datos/Datos procesados"

# Crear la carpeta de destino si no existe
os.makedirs(carpeta_destino, exist_ok=True)

archivos = [
    "GRD_CLASIFICADO_2019.csv",
    "GRD_CLASIFICADO_2020.csv",
    "GRD_CLASIFICADO_2021.csv",
    "GRD_CLASIFICADO_2022.csv",
    "GRD_CLASIFICADO_2023.csv",
    "GRD_CLASIFICADO_2024.csv"
]

## **Análisis de Resultados de Procesamiento**

### **1. Integridad de la Muestra (Pérdida de Datos)**
La política de eliminación estricta de variables objetivo nulas o con la etiqueta "NO IDENTIFICADA" resultó en una pérdida de datos estadísticamente marginal. A lo largo del periodo 2019-2024, la merma máxima anual fue de apenas 80 filas (año 2019) sobre un volumen superior a 1.1 millones de episodios. Esta retención del **~99.99%** de los datos garantiza que no se han introducido sesgos de selección.

In [2]:
# ============================================================================
# BLOQUE 1: PROCESAMIENTO SIMULTÁNEO DE TARGETS
# ============================================================================
print("Iniciando procesamiento de variables objetivo (Targets)...\n" + "="*60)

for archivo in archivos:
    ruta_origen = os.path.join(carpeta_origen, archivo)
    
    # Nuevo nombre para el archivo procesado
    año = archivo[-8:-4]
    nuevo_nombre = f"GRD_PROCESADO_{año}.csv"
    ruta_destino = os.path.join(carpeta_destino, nuevo_nombre)
    
    # 1. Leer el archivo clasificado original
    df = pd.read_csv(ruta_origen, low_memory=False)
    filas_iniciales = len(df)
    
    # --- TARGET 1: MORTALIDAD ---
    # Eliminar nulos/desconocidos
    df = df[~df["TIPOALTA"].isin(["NO IDENTIFICADA", "DESCONOCIDO", "nan", "NaN"])]
    df = df[df["TIPOALTA"].notnull()]
    # Binarizar: 1 = Fallecido, 0 = Vivo (Resto)
    df["MORTALIDAD"] = df["TIPOALTA"].apply(lambda x: 1 if str(x).strip().upper() == "FALLECIDO" else 0)
    
    # --- TARGET 2: SEVERIDAD ---
    df["IR_29301_SEVERIDAD"] = df["IR_29301_SEVERIDAD"].astype(str).str.strip()
    # Filtrar solo categorías válidas (elimina nulos y desconocidos)
    df = df[df["IR_29301_SEVERIDAD"].isin(["0", "1", "2", "3"])]
    df["SEVERIDAD"] = df["IR_29301_SEVERIDAD"].astype(int)
    
    # --- TARGET 3: CONSUMO DE RECURSOS ---
    # Limpiar formato de peso relativo
    peso_limpio = df["IR_29301_PESO"].astype(str).str.replace(",", ".", regex=False)
    df["PESO_NUMERICO"] = pd.to_numeric(peso_limpio, errors="coerce")
    # Eliminar nulos en el peso
    df = df[df["PESO_NUMERICO"].notnull()]
    
    # Discretizar en Terciles (0: Bajo, 1: Medio, 2: Alto)
    try:
        df["CONSUMO_RECURSOS"] = pd.qcut(df["PESO_NUMERICO"], q=3, labels=[0, 1, 2])
    except ValueError:
        print(f"  [!] Error al calcular terciles en {año}.")
        df["CONSUMO_RECURSOS"] = None

    # Eliminar cualquier fila que haya quedado sin asignar en el tercil
    df = df[df["CONSUMO_RECURSOS"].notnull()]
    
    # 2. Guardar archivo procesado
    df.to_csv(ruta_destino, index=False, encoding="utf-8-sig")
    
    # 3. Reporte de pérdida de datos
    filas_finales = len(df)
    filas_perdidas = filas_iniciales - filas_finales
    pct_perdida = (filas_perdidas / filas_iniciales) * 100
    
    print(f"[{año}] Archivo guardado como {nuevo_nombre}")
    print(f"  -> Filas iniciales: {filas_iniciales:,}")
    print(f"  -> Filas eliminadas por Target Nulo: {filas_perdidas:,} ({pct_perdida:.2f}%)")
    print(f"  -> Filas finales útiles: {filas_finales:,}\n")

print("="*60 + "\n¡Procesamiento de Targets Finalizado!")

Iniciando procesamiento de variables objetivo (Targets)...
[2019] Archivo guardado como GRD_PROCESADO_2019.csv
  -> Filas iniciales: 1,151,398
  -> Filas eliminadas por Target Nulo: 80 (0.01%)
  -> Filas finales útiles: 1,151,318

[2020] Archivo guardado como GRD_PROCESADO_2020.csv
  -> Filas iniciales: 781,911
  -> Filas eliminadas por Target Nulo: 23 (0.00%)
  -> Filas finales útiles: 781,888

[2021] Archivo guardado como GRD_PROCESADO_2021.csv
  -> Filas iniciales: 816,909
  -> Filas eliminadas por Target Nulo: 0 (0.00%)
  -> Filas finales útiles: 816,909

[2022] Archivo guardado como GRD_PROCESADO_2022.csv
  -> Filas iniciales: 932,837
  -> Filas eliminadas por Target Nulo: 23 (0.00%)
  -> Filas finales útiles: 932,814

[2023] Archivo guardado como GRD_PROCESADO_2023.csv
  -> Filas iniciales: 1,039,587
  -> Filas eliminadas por Target Nulo: 30 (0.00%)
  -> Filas finales útiles: 1,039,557

[2024] Archivo guardado como GRD_PROCESADO_2024.csv
  -> Filas iniciales: 1,085,813
  -> Filas

### **2. Contraste Epidemiológico: Mortalidad y Severidad**
Las tablas de frecuencia segmentadas evidencian la carga clínica de la cohorte oncológica:
* **Mortalidad Intrahospitalaria:** Mientras la población de control mantiene una tasa histórica de mortalidad que oscila entre el 2.1% y 3.7%, la cohorte oncológica presenta tasas consistentes entre el **5.4% y 6.9%**. El paciente con cáncer tiene aproximadamente el doble de riesgo de fallecer durante el episodio.
* **Desplazamiento de la Severidad:** En la población general, la gravedad se concentra en los niveles 1 (Menor) y 2 (Moderada). En contraste, el perfil oncológico sufre un desplazamiento severo hacia la derecha: para 2024, casi **30.000 episodios oncológicos se clasificaron en Gravedad 3 (Mayor)**, superando ampliamente a los episodios sin gravedad (11.000).

In [3]:
def tabla_frecuencias_target(carpeta, archivos_proc, columna):
    """
    Genera tablas de frecuencias anuales para una variable objetivo,
    segmentada por cohorte: Total, Oncológica y Control.
    """
    res_total, res_onco, res_control = {}, {}, {}
    
    for archivo in archivos_proc:
        ruta = os.path.join(carpeta, archivo)
        año = archivo[-8:-4]
        
        # Leer solo las columnas necesarias para no saturar memoria
        df = pd.read_csv(ruta, usecols=[columna, 'CATEGORIA_CANCER'], low_memory=False)
        
        # Filtrar cohortes
        mask_control = df['CATEGORIA_CANCER'] == 'SIN_CANCER'
        df_onco = df[~mask_control]
        df_control = df[mask_control]
        
        # Calcular frecuencias
        res_total[año] = df[columna].value_counts()
        res_onco[año] = df_onco[columna].value_counts()
        res_control[año] = df_control[columna].value_counts()

    # Formatear tablas
    tabla_total = pd.DataFrame(res_total).fillna(0).astype(int).sort_index()
    tabla_onco = pd.DataFrame(res_onco).fillna(0).astype(int).sort_index()
    tabla_control = pd.DataFrame(res_control).fillna(0).astype(int).sort_index()
    
    return tabla_total, tabla_onco, tabla_control

In [4]:
archivos_procesados = [f"GRD_PROCESADO_{año}.csv" for año in range(2019, 2025)]

def mostrar_resultados(variable):
    df_total, df_onco, df_control = tabla_frecuencias_target(carpeta_destino, archivos_procesados, variable)
    display(f"Tabla de frecuencias para {variable} - Cohorte Total:")
    display(df_total)
    display(f"Tabla de frecuencias para {variable} - Cohorte Oncológica:")
    display(df_onco)
    display(f"Tabla de frecuencias para {variable} - Cohorte Control:")
    display(df_control)

#### **a) Mortalidad:** 0 vivo, 1 fallecido.
- **Decisión de procesamiento:** Binarizar y eliminar casos nulos (no identificados). En específico la clase positiva (1 - fallecido) se va a componer por los casos de la categoría FALLECIDO, y la clase negativa (0 - vivo) se va a componer por el resto de destinos del paciente (DOMICILIO, HOSPITALIZACION DOMICILIARIA, DERIVACION OTRO HOSPITAL DEL SERVICIO, DERIVACION OTRO HOSPITAL DE LA RED NACIONAL, ALTA VOLUNTARIA, DERIVACION INST. PRIVADA, DERIVACION A OTROS CENTROS, FUGA DEL PACIENTE, DERIVACION INST. PRIVADA (VOLUNTARIO)), exceptuando los casos desconocidos (NO IDENTIFICADA).

In [5]:
# Análisis de mortalidad
mostrar_resultados("MORTALIDAD")

'Tabla de frecuencias para MORTALIDAD - Cohorte Total:'

,2019,2020,2021,2022,2023,2024
MORTALIDAD,,,,,,
0,1121734,751946,784944,905453,1014435,1059124
1,29584,29942,31965,27361,25122,26674


'Tabla de frecuencias para MORTALIDAD - Cohorte Oncológica:'

,2019,2020,2021,2022,2023,2024
MORTALIDAD,,,,,,
0,92503,60463,63503,73218,85401,93144
1,5321,4462,4423,4849,5314,6007


'Tabla de frecuencias para MORTALIDAD - Cohorte Control:'

,2019,2020,2021,2022,2023,2024
MORTALIDAD,,,,,,
0,1029231,691483,721441,832235,929034,965980
1,24263,25480,27542,22512,19808,20667


#### **b) Severidad:**
- **Decisión de procesamiento:** Al igual que en el caso de inmortalidad, los casos DESCONOCIDO serán eliminados, para no imputar datos en las variables objetivo

In [6]:
# Análisis de severidad
mostrar_resultados("SEVERIDAD")

'Tabla de frecuencias para SEVERIDAD - Cohorte Total:'

,2019,2020,2021,2022,2023,2024
SEVERIDAD,,,,,,
0,202516,79664,102938,154772,193550,212051
1,558541,366278,340136,373440,388694,387258
2,242270,176549,183822,213260,253086,268865
3,147991,159397,190013,191342,204227,217624


'Tabla de frecuencias para SEVERIDAD - Cohorte Oncológica:'

,2019,2020,2021,2022,2023,2024
SEVERIDAD,,,,,,
0,20675,3648,5689,7909,10215,11081
1,26703,19058,19126,20488,21770,23198
2,30675,23807,23718,26710,31917,34607
3,19771,18412,19393,22960,26813,30265


'Tabla de frecuencias para SEVERIDAD - Cohorte Control:'

,2019,2020,2021,2022,2023,2024
SEVERIDAD,,,,,,
0,181841,76016,97249,146863,183335,200970
1,531838,347220,321010,352952,366924,364060
2,211595,152742,160104,186550,221169,234258
3,128220,140985,170620,168382,177414,187359


### **3. Consumo de Recursos (Estratificación por Terciles)**
La estrategia de discretizar el consumo de recursos (`IR_29301_PESO`) calculando los terciles (*q-cut*) sobre la **Población Total** fue metodológicamente exitosa. En la cohorte total, las clases quedaron perfectamente balanceadas (aprox. 33% en cada tercil). Sin embargo, al observar cómo se distribuye la cohorte oncológica dentro de estos umbrales globales, el hallazgo es contundente:
* Para el año 2024, de ~99.000 episodios oncológicos, **más de 57.000 (cerca del 58%) cayeron en el tercil de ALTO consumo**. 
* Solo ~10.000 episodios (10%) calificaron como de bajo consumo.
* Esto comprueba cuantitativamente que la patología oncológica es uno de los principales motores de presión financiera y operativa sobre la red pública de salud.

In [7]:
# Análisis de consumo de recursos
mostrar_resultados("CONSUMO_RECURSOS")

'Tabla de frecuencias para CONSUMO_RECURSOS - Cohorte Total:'

,2019,2020,2021,2022,2023,2024
CONSUMO_RECURSOS,,,,,,
0,384662,266705,272677,316599,347755,362300
1,385155,256653,273616,306011,345834,365453
2,381501,258530,270616,310204,345968,358045


'Tabla de frecuencias para CONSUMO_RECURSOS - Cohorte Oncológica:'

,2019,2020,2021,2022,2023,2024
CONSUMO_RECURSOS,,,,,,
0,10304,4463,5948,7038,8998,10188
1,21605,22658,24915,20558,23684,31300
2,65915,37804,37063,50471,58033,57663


'Tabla de frecuencias para CONSUMO_RECURSOS - Cohorte Control:'

,2019,2020,2021,2022,2023,2024
CONSUMO_RECURSOS,,,,,,
0,374358,262242,266729,309561,338757,352112
1,363550,233995,248701,285453,322150,334153
2,315586,220726,233553,259733,287935,300382
